# GR00T-N1.6-3B SageMaker 파인튜닝 & 배포 End-to-End 파이프라인

이 노트북은 NVIDIA GR00T-N1.6-3B Vision-Language-Action(VLA) 모델을 AWS SageMaker에서
파인튜닝하고 추론 엔드포인트로 배포하는 **전체 워크플로우**를 단계별로 실행합니다.

### 워크플로우 개요

| 단계 | 설명 | 예상 시간 |
|------|------|----------|
| 1. 환경 설정 | 패키지 설치, AWS 자격증명 확인 | ~2분 |
| 2. 샘플 데이터셋 생성 | LeRobot v2 호환 테스트 데이터 | ~1분 |
| 3. S3 업로드 | 데이터셋 + 더미 모델 아티팩트 | ~5분 |
| 4. ECR 컨테이너 빌드 | 학습/추론 Docker 이미지 | ~15-30분 |
| 5. 로컬 검증 | 설정 및 핸들러 테스트 | ~2분 |
| 6. SageMaker 학습 | 파인튜닝 Training Job | ~30-60분 |
| 7. 엔드포인트 배포 | 추론 서비스 생성 | ~5-10분 |
| 8. 추론 테스트 | 엔드포인트 호출 | ~1분 |
| 9. 리소스 정리 | 엔드포인트 삭제 | ~2분 |



> ⚠️ **비용 주의**: 이 노트북은 실제 AWS 리소스를 생성합니다.
> 학습 인스턴스(`ml.p4d.24xlarge` ~$32.77/hr), 추론 인스턴스(`ml.g5.2xlarge` ~$1.52/hr)는
> 사용 시간에 따라 비용이 발생합니다. 반드시 마지막 **리소스 정리** 단계를 실행하세요.


---
## Step 1: 환경 설정

먼저 필요한 Python 패키지를 설치하고, AWS 자격증명이 올바르게 구성되어 있는지 확인합니다.

**왜 이 패키지들이 필요한가?**
- `boto3` — AWS 서비스(S3, SageMaker, ECR)와 프로그래밍 방식으로 상호작용
- `sagemaker` — SageMaker Training Job과 Endpoint를 Python SDK로 관리
- `pandas`, `pyarrow` — LeRobot v2 데이터셋이 Parquet 형식이므로 데이터 생성/읽기에 필요
- `Pillow` — 테스트용 이미지 생성 및 처리


In [ ]:
# 필수 패키지 설치
!pip install -q boto3 sagemaker pandas pyarrow Pillow numpy


### 1.1 AWS 자격증명 확인

`aws configure`가 완료되어 있어야 합니다. 아래 셀에서 현재 AWS 계정 정보와 리전을 확인합니다.
이 값들은 이후 모든 단계에서 변수로 저장되어 자동으로 사용됩니다.


In [ ]:
import boto3
import sagemaker
import json
import os
import sys
import time
import subprocess
import base64
import numpy as np
from datetime import datetime
from pathlib import Path

# AWS 세션 초기화
boto_session = boto3.Session()
sm_session = sagemaker.Session(boto_session=boto_session)

# 계정 정보 자동 감지
REGION = boto_session.region_name
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

print(f'AWS Account ID : {ACCOUNT_ID}')
print(f'AWS Region     : {REGION}')
print(f'SageMaker SDK  : {sagemaker.__version__}')


### 1.2 핵심 변수 설정

아래 변수들은 전체 파이프라인에서 일관되게 사용됩니다.
`ROLE_ARN`만 본인의 SageMaker 실행 역할 ARN으로 변경하세요.

**SageMaker 실행 역할이란?**
SageMaker Training Job과 Endpoint가 S3에서 데이터를 읽고, ECR에서 컨테이너를 가져오고,
CloudWatch에 로그를 쓰기 위해 필요한 IAM 역할입니다.
`AmazonSageMakerFullAccess` 정책이 연결되어 있어야 합니다.


In [ ]:
# ============================================================
# ⚡ 여기만 수정하세요
# ============================================================
ROLE_ARN = 'arn:aws:iam::<YOUR_ACCOUNT_ID>:role/<YOUR_SAGEMAKER_ROLE>'
# 예시: ROLE_ARN = 'arn:aws:iam::123456789012:role/SageMakerExecutionRole'

# 자동 감지 시도 (SageMaker 노트북 인스턴스에서 실행 시)
if '<YOUR_ACCOUNT_ID>' in ROLE_ARN:
    try:
        ROLE_ARN = sagemaker.get_execution_role()
        print(f'SageMaker 실행 역할 자동 감지: {ROLE_ARN}')
    except ValueError:
        print('⚠️  ROLE_ARN을 직접 설정해주세요!')
        print('    예: arn:aws:iam::123456789012:role/SageMakerExecutionRole')

# 프로젝트 공통 변수
PROJECT_PREFIX = 'groot-n16'
TIMESTAMP = datetime.utcnow().strftime('%Y%m%d-%H%M%S')
S3_BUCKET = sm_session.default_bucket()  # SageMaker 기본 버킷 사용

# S3 경로 (이후 단계에서 자동으로 참조됨)
S3_DATASET_URI = f's3://{S3_BUCKET}/{PROJECT_PREFIX}/datasets/sample-lerobot-v2'
S3_MODEL_URI = f's3://{S3_BUCKET}/{PROJECT_PREFIX}/models/groot-n16-base'
S3_OUTPUT_URI = f's3://{S3_BUCKET}/{PROJECT_PREFIX}/output/{TIMESTAMP}'

# ECR 리포지토리 이름
ECR_TRAINING_REPO = f'{PROJECT_PREFIX}-training'
ECR_INFERENCE_REPO = f'{PROJECT_PREFIX}-inference'

# 엔드포인트 이름
ENDPOINT_NAME = f'{PROJECT_PREFIX}-endpoint-{TIMESTAMP}'

print('=' * 60)
print('파이프라인 설정 요약')
print('=' * 60)
print(f'Role ARN       : {ROLE_ARN}')
print(f'S3 Bucket      : {S3_BUCKET}')
print(f'Dataset S3     : {S3_DATASET_URI}')
print(f'Model S3       : {S3_MODEL_URI}')
print(f'Output S3      : {S3_OUTPUT_URI}')
print(f'Training ECR   : {ECR_TRAINING_REPO}')
print(f'Inference ECR  : {ECR_INFERENCE_REPO}')
print(f'Endpoint Name  : {ENDPOINT_NAME}')
print('=' * 60)


---
## Step 2: LeRobot v2 데이터셋 준비

GR00T-N1.6 파인튜닝에는 **LeRobot v2 호환 형식**의 데이터셋이 필요합니다.
아래 두 가지 옵션 중 하나를 선택하세요:

| 옵션 | 방법 | 장점 | 적합한 경우 |
|------|------|------|------------|
| **A. HuggingFace 데이터셋** | 공개 LeRobot 데이터셋 다운로드 | 실제 로봇 데이터, 바로 사용 가능 | 파이프라인 검증, 빠른 실험 |
| **B. 합성 데이터 생성** | 랜덤 데이터로 직접 생성 | 오프라인 가능, 구조 이해 | 데이터 형식 학습, 네트워크 없는 환경 |

> 💡 **옵션 A를 추천합니다.** 실제 로봇 조작 데이터로 파이프라인을 검증할 수 있습니다.
> 시뮬레이션 데이터가 필요하면 [Isaac Sim 연동 가이드](../guide/ISAAC_SIM_GUIDE.md)를 참고하세요.


### 옵션 A: HuggingFace LeRobot 데이터셋 다운로드 (추천)

HuggingFace Hub에는 다양한 LeRobot v2 호환 데이터셋이 공개되어 있습니다.
여기서는 `lerobot/pusht` 데이터셋을 사용합니다.

**사용 가능한 주요 데이터셋:**

| 데이터셋 | 설명 | 에피소드 수 | 크기 |
|----------|------|------------|------|
| `lerobot/pusht` | T자 블록 밀기 (2D) | ~200 | ~50MB |
| `lerobot/aloha_sim_transfer_cube_human` | ALOHA 큐브 옮기기 | ~50 | ~500MB |
| `lerobot/xarm_lift_medium_replay` | xArm 물체 들기 | ~100 | ~200MB |

**왜 HuggingFace 데이터셋인가?**
- 이미 LeRobot v2 형식으로 정리되어 있어 변환 작업 불필요
- 실제 로봇/시뮬레이션 데이터이므로 학습 결과가 더 의미 있음
- `huggingface_hub` 라이브러리로 한 줄이면 다운로드 가능


In [ ]:
# HuggingFace Hub 설치
!pip install -q huggingface_hub datasets


In [ ]:
from huggingface_hub import snapshot_download

# ============================================================
# 데이터셋 선택 — 원하는 데이터셋으로 변경 가능
# ============================================================
HF_DATASET_REPO = 'lerobot/pusht'  # 가볍고 빠른 테스트용
# HF_DATASET_REPO = 'lerobot/aloha_sim_transfer_cube_human'  # ALOHA 시뮬레이션
# HF_DATASET_REPO = 'lerobot/xarm_lift_medium_replay'        # xArm 로봇

LOCAL_DATASET_DIR = f'/tmp/lerobot-{HF_DATASET_REPO.split("/")[-1]}'

print(f'📥 HuggingFace 데이터셋 다운로드: {HF_DATASET_REPO}')
print(f'   저장 경로: {LOCAL_DATASET_DIR}')
print()

snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type='dataset',
    local_dir=LOCAL_DATASET_DIR,
)

print(f'\n✅ 다운로드 완료: {LOCAL_DATASET_DIR}')


In [ ]:
# 다운로드된 데이터셋 구조 확인
print(f'📂 데이터셋 구조: {HF_DATASET_REPO}')
for root, dirs, files in os.walk(LOCAL_DATASET_DIR):
    # .git 디렉토리 제외
    dirs[:] = [d for d in dirs if d != '.git']
    level = root.replace(LOCAL_DATASET_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        subindent = '  ' * (level + 1)
        for file in sorted(files)[:5]:
            fpath = os.path.join(root, file)
            size_kb = os.path.getsize(fpath) / 1024
            print(f'{subindent}{file} ({size_kb:.1f} KB)')
        if len(files) > 5:
            print(f'{subindent}... ({len(files) - 5}개 더)')

# meta/info.json 내용 확인
info_path = os.path.join(LOCAL_DATASET_DIR, 'meta', 'info.json')
if os.path.exists(info_path):
    with open(info_path) as f:
        info = json.load(f)
    print(f'\n📊 데이터셋 정보:')
    print(f'   총 에피소드: {info.get("total_episodes", "N/A")}')
    print(f'   총 프레임: {info.get("total_frames", "N/A")}')
    print(f'   FPS: {info.get("fps", "N/A")}')
    print(f'   로봇 타입: {info.get("robot_type", "N/A")}')
else:
    print('\n⚠️  meta/info.json을 찾을 수 없습니다')

print(f'\n✅ 옵션 A 완료 — 아래 옵션 B는 건너뛰고 Step 3으로 이동하세요')


### 옵션 B: 합성 샘플 데이터셋 직접 생성

HuggingFace 접근이 어렵거나, 데이터셋 형식을 직접 이해하고 싶다면
아래 셀들을 실행하여 합성 데이터를 생성할 수 있습니다.

> ⚠️ **옵션 A를 실행했다면 이 섹션은 건너뛰세요.**


In [ ]:
import pandas as pd
from PIL import Image
import io

# 로컬 샘플 데이터셋 경로
LOCAL_DATASET_DIR = '/tmp/groot-sample-dataset'

# 데이터셋 파라미터
NUM_EPISODES = 3
FRAMES_PER_EPISODE = 20
ACTION_DIM = 7        # 7-DoF 로봇 팔 (x, y, z, rx, ry, rz, gripper)
STATE_DIM = 7         # 고유수용감각 벡터 차원
IMAGE_SIZE = (224, 224)  # GR00T 입력 이미지 크기

print(f'샘플 데이터셋 생성 중...')
print(f'  에피소드 수: {NUM_EPISODES}')
print(f'  에피소드당 프레임: {FRAMES_PER_EPISODE}')
print(f'  액션 차원: {ACTION_DIM}')
print(f'  이미지 크기: {IMAGE_SIZE}')


In [ ]:
# 디렉토리 구조 생성
for subdir in ['meta', 'data/chunk-000', 'videos/chunk-000']:
    os.makedirs(f'{LOCAL_DATASET_DIR}/{subdir}', exist_ok=True)

# --- meta/info.json ---
# 데이터셋의 전체 메타데이터를 정의합니다.
# embodiment_tag는 로봇 종류를 식별하는 키로, 학습 시 --embodiment-tag와 일치해야 합니다.
info = {
    'codebase_version': 'v2.0',
    'robot_type': 'sample_robot',
    'total_episodes': NUM_EPISODES,
    'total_frames': NUM_EPISODES * FRAMES_PER_EPISODE,
    'fps': 10,
    'features': {
        'observation.images.top': {
            'dtype': 'image',
            'shape': [IMAGE_SIZE[0], IMAGE_SIZE[1], 3],
            'names': ['height', 'width', 'channels']
        },
        'observation.state': {
            'dtype': 'float32',
            'shape': [STATE_DIM],
            'names': ['state']
        },
        'action': {
            'dtype': 'float32',
            'shape': [ACTION_DIM],
            'names': ['action']
        }
    }
}

with open(f'{LOCAL_DATASET_DIR}/meta/info.json', 'w') as f:
    json.dump(info, f, indent=2)
print('✅ meta/info.json 생성 완료')


In [ ]:
# --- 에피소드 데이터 생성 ---
# 각 에피소드는 Parquet 파일로 저장됩니다.
# 실제 데이터에서는 로봇의 관측(이미지, 상태)과 액션이 시간순으로 기록됩니다.

episodes_meta = []

for ep_idx in range(NUM_EPISODES):
    # 에피소드별 비디오 프레임 디렉토리
    ep_video_dir = f'{LOCAL_DATASET_DIR}/videos/chunk-000/episode_{ep_idx:06d}'
    os.makedirs(ep_video_dir, exist_ok=True)
    
    rows = []
    for frame_idx in range(FRAMES_PER_EPISODE):
        # 랜덤 상태 벡터 (관절 각도 시뮬레이션)
        state = np.random.uniform(-1.0, 1.0, size=STATE_DIM).tolist()
        # 랜덤 액션 벡터 (관절 속도 시뮬레이션)
        action = np.random.uniform(-0.5, 0.5, size=ACTION_DIM).tolist()
        
        # 합성 RGB 이미지 생성 (색상이 프레임마다 변하도록)
        r = int(255 * frame_idx / FRAMES_PER_EPISODE)
        g = int(255 * (1 - frame_idx / FRAMES_PER_EPISODE))
        b = int(128 + 127 * np.sin(frame_idx * 0.5))
        img = Image.new('RGB', IMAGE_SIZE, (r, g, b))
        img.save(f'{ep_video_dir}/frame_{frame_idx:06d}.png')
        
        rows.append({
            'episode_index': ep_idx,
            'frame_index': frame_idx,
            'timestamp': frame_idx / 10.0,  # 10 FPS
            'observation.state': state,
            'action': action,
        })
    
    # Parquet 파일로 저장
    df = pd.DataFrame(rows)
    parquet_path = f'{LOCAL_DATASET_DIR}/data/chunk-000/episode_{ep_idx:06d}.parquet'
    df.to_parquet(parquet_path, index=False)
    
    episodes_meta.append({
        'episode_index': ep_idx,
        'length': FRAMES_PER_EPISODE,
    })
    print(f'  에피소드 {ep_idx}: {FRAMES_PER_EPISODE} 프레임, Parquet + {FRAMES_PER_EPISODE} 이미지 저장')

# --- meta/episodes.jsonl ---
with open(f'{LOCAL_DATASET_DIR}/meta/episodes.jsonl', 'w') as f:
    for ep in episodes_meta:
        f.write(json.dumps(ep) + '\n')
print('✅ meta/episodes.jsonl 생성 완료')

# --- meta/stats.json ---
# 데이터 정규화에 사용되는 통계 정보
stats = {
    'observation.state': {
        'mean': [0.0] * STATE_DIM,
        'std': [1.0] * STATE_DIM,
        'min': [-1.0] * STATE_DIM,
        'max': [1.0] * STATE_DIM,
    },
    'action': {
        'mean': [0.0] * ACTION_DIM,
        'std': [0.5] * ACTION_DIM,
        'min': [-0.5] * ACTION_DIM,
        'max': [0.5] * ACTION_DIM,
    }
}
with open(f'{LOCAL_DATASET_DIR}/meta/stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('✅ meta/stats.json 생성 완료')
print(f'\n📁 샘플 데이터셋 경로: {LOCAL_DATASET_DIR}')


### 2.1 데이터셋 검증

생성된 데이터셋이 올바른 형식인지 확인합니다.
실제 데이터를 사용할 때도 이 검증 단계를 거치면 학습 시 데이터 형식 오류를 미리 방지할 수 있습니다.


In [ ]:
# 데이터셋 구조 확인
print('📂 데이터셋 구조:')
for root, dirs, files in os.walk(LOCAL_DATASET_DIR):
    level = root.replace(LOCAL_DATASET_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:  # 너무 깊은 디렉토리는 생략
        subindent = '  ' * (level + 1)
        for file in files[:5]:  # 파일이 많으면 5개만 표시
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... ({len(files) - 5}개 더)')

# Parquet 파일 내용 확인
print('\n📊 에피소드 0 데이터 미리보기:')
sample_df = pd.read_parquet(f'{LOCAL_DATASET_DIR}/data/chunk-000/episode_000000.parquet')
print(f'  행 수: {len(sample_df)}')
print(f'  컬럼: {list(sample_df.columns)}')
print(sample_df.head(3).to_string())

# 이미지 확인
sample_img = Image.open(f'{LOCAL_DATASET_DIR}/videos/chunk-000/episode_000000/frame_000000.png')
print(f'\n🖼️  샘플 이미지 크기: {sample_img.size}, 모드: {sample_img.mode}')


---
## Step 3: S3에 데이터 업로드

SageMaker Training Job은 S3에서 데이터를 읽어옵니다.
생성한 샘플 데이터셋과 더미 모델 아티팩트를 S3에 업로드합니다.

**왜 S3를 사용하는가?**
- SageMaker Training Job은 학습 시작 시 S3에서 데이터를 컨테이너 내부로 자동 복사합니다
- 학습 완료 후 모델 아티팩트도 S3에 자동 저장됩니다
- 이 구조 덕분에 컨테이너는 stateless하게 유지되고, 데이터는 영구 저장됩니다

> 💡 **실제 사용 시**: 이 단계에서 HuggingFace에서 다운로드한 GR00T-N1.6-3B 모델을 업로드합니다.
> ```bash
> git lfs install
> git clone https://huggingface.co/nvidia/GR00T-N1.6-3B
> aws s3 cp --recursive ./GR00T-N1.6-3B s3://bucket/models/groot-n16/
> ```


In [ ]:
# 데이터셋을 S3에 업로드
print(f'📤 데이터셋 업로드 중: {LOCAL_DATASET_DIR} → {S3_DATASET_URI}')
!aws s3 cp --recursive {LOCAL_DATASET_DIR} {S3_DATASET_URI} --quiet
print(f'✅ 데이터셋 업로드 완료')

# S3에 업로드된 파일 확인
print(f'\n📂 S3 데이터셋 내용:')
!aws s3 ls {S3_DATASET_URI}/ --recursive --summarize | tail -5


### 3.1 더미 모델 아티팩트 업로드

실제로는 HuggingFace에서 GR00T-N1.6-3B 모델(~6GB)을 다운로드하여 업로드합니다.
여기서는 파이프라인 테스트를 위해 더미 `config.json`을 업로드합니다.

**모델 아티팩트란?**
- 사전학습된 모델의 가중치, 설정 파일, 토크나이저 등을 포함하는 파일 세트
- SageMaker는 이를 학습 컨테이너의 `/opt/ml/input/data/model/` 경로에 마운트합니다


In [ ]:
# 더미 모델 config 생성 및 업로드
LOCAL_MODEL_DIR = '/tmp/groot-dummy-model'
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

dummy_config = {
    'model_type': 'groot-n16',
    'model_name': 'GR00T-N1.6-3B',
    'hidden_size': 2048,
    'num_attention_heads': 16,
    'num_hidden_layers': 24,
    'action_dim': ACTION_DIM,
    'note': 'This is a dummy config for pipeline testing'
}

with open(f'{LOCAL_MODEL_DIR}/config.json', 'w') as f:
    json.dump(dummy_config, f, indent=2)

# 더미 가중치 파일 (실제로는 수 GB)
np.save(f'{LOCAL_MODEL_DIR}/dummy_weights.npy', np.random.randn(100, 100).astype(np.float32))

print(f'📤 모델 아티팩트 업로드 중: {LOCAL_MODEL_DIR} → {S3_MODEL_URI}')
!aws s3 cp --recursive {LOCAL_MODEL_DIR} {S3_MODEL_URI} --quiet
print(f'✅ 모델 아티팩트 업로드 완료')

# 확인
print(f'\n📂 S3 모델 내용:')
!aws s3 ls {S3_MODEL_URI}/


---
## Step 4: Docker 컨테이너 빌드 & ECR 푸시

SageMaker는 사용자가 제공한 Docker 컨테이너 안에서 학습과 추론을 실행합니다.
GR00T-N1.6에는 CUDA 12.4, PyTorch, flash-attn, Isaac-GR00T 등 특수한 의존성이 필요하므로
커스텀 컨테이너를 빌드합니다.

### 컨테이너 구성

| 컨테이너 | 베이스 이미지 | 주요 의존성 | 용도 |
|----------|-------------|-----------|------|
| Training | `nvidia/cuda:12.4.1-devel` | PyTorch, flash-attn, Isaac-GR00T[train] | 파인튜닝 |
| Inference | `nvidia/cuda:12.4.1-runtime` | PyTorch, flash-attn, Isaac-GR00T, MMS | 추론 서빙 |

**왜 커스텀 컨테이너인가?**
- SageMaker 기본 이미지에는 Isaac-GR00T와 flash-attn이 포함되어 있지 않음
- CUDA 12.4 + PyTorch 2.6 조합이 필요
- 학습 컨테이너는 `devel` 이미지(컴파일 도구 포함), 추론은 `runtime` 이미지(경량)를 사용

> ⏱️ **소요 시간**: 첫 빌드 시 15-30분 소요됩니다 (flash-attn 컴파일 포함).
> 이후 빌드는 Docker 캐시 덕분에 훨씬 빠릅니다.


### 4.1 학습용 컨테이너 빌드

`build_and_push.sh` 스크립트가 다음을 자동으로 수행합니다:
1. ECR에 Docker 인증
2. ECR 리포지토리 생성 (없으면)
3. Dockerfile 기반 이미지 빌드
4. ECR에 태그 & 푸시

빌드 완료 후 출력되는 **Image URI**가 `TRAINING_IMAGE_URI` 변수에 저장됩니다.


In [ ]:
# 프로젝트 루트 디렉토리 (build_and_push.sh는 sagemaker-vla/ 에서 실행해야 함)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(PROJECT_ROOT) != 'sagemaker-vla':
    # 노트북이 sagemaker-vla/notebook/ 에 있는 경우
    if os.path.exists(os.path.join(os.getcwd(), '..', 'scripts', 'build_and_push.sh')):
        PROJECT_ROOT = os.path.abspath('..')
    elif os.path.exists(os.path.join(os.getcwd(), 'sagemaker-vla', 'scripts', 'build_and_push.sh')):
        PROJECT_ROOT = os.path.abspath('sagemaker-vla')
    else:
        PROJECT_ROOT = os.path.abspath('.')

print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'Dockerfile 확인: {os.path.exists(os.path.join(PROJECT_ROOT, "docker", "Dockerfile.training"))}')


In [ ]:
# 학습용 컨테이너 빌드 & ECR 푸시
print('🔨 학습용 컨테이너 빌드 시작...')
print(f'   리전: {REGION}, 계정: {ACCOUNT_ID}')
print()

!cd {PROJECT_ROOT} && bash scripts/build_and_push.sh \
    --type training \
    --account-id {ACCOUNT_ID} \
    --region {REGION}

# ECR Image URI 변수 저장 (이후 단계에서 사용)
TRAINING_IMAGE_URI = f'{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_TRAINING_REPO}:latest'
print(f'\n📦 Training Image URI: {TRAINING_IMAGE_URI}')


### 4.2 추론용 컨테이너 빌드

추론 컨테이너는 멀티스테이지 빌드를 사용합니다:
- **Stage 1 (builder)**: CUDA devel 이미지에서 flash-attn 컴파일
- **Stage 2 (runtime)**: CUDA runtime 이미지에 필요한 파일만 복사 → 최종 이미지 크기 최소화

추론 컨테이너에는 MMS(Multi-Model Server)가 포함되어 있어
SageMaker가 요구하는 `/ping`(헬스체크)과 `/invocations`(추론) 엔드포인트를 자동으로 제공합니다.


In [ ]:
# 추론용 컨테이너 빌드 & ECR 푸시
print('🔨 추론용 컨테이너 빌드 시작...')
print(f'   리전: {REGION}, 계정: {ACCOUNT_ID}')
print()

!cd {PROJECT_ROOT} && bash scripts/build_and_push.sh \
    --type inference \
    --account-id {ACCOUNT_ID} \
    --region {REGION}

# ECR Image URI 변수 저장
INFERENCE_IMAGE_URI = f'{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_INFERENCE_REPO}:latest'
print(f'\n📦 Inference Image URI: {INFERENCE_IMAGE_URI}')


---
## Step 5: 로컬 검증

SageMaker에 작업을 제출하기 전에, 로컬에서 설정과 코드가 올바른지 검증합니다.
이 단계를 통해 비용이 드는 클라우드 작업에서 발생할 수 있는 오류를 미리 잡을 수 있습니다.

### 검증 항목
1. **인스턴스 타입 검증** — VRAM 요구사항 충족 여부
2. **S3 URI 형식 검증** — 올바른 S3 경로 형식
3. **추론 핸들러 테스트** — input_fn, predict_fn, output_fn 동작 확인
4. **ECR 이미지 존재 확인** — 빌드된 이미지가 ECR에 있는지


In [ ]:
# src 모듈을 import할 수 있도록 경로 추가
sys.path.insert(0, PROJECT_ROOT)

from src.config import (
    validate_instance_type,
    validate_s3_uri,
    get_instance_recommendations,
    TrainingConfig,
    DeploymentConfig,
)

print('=' * 60)
print('🔍 로컬 검증 시작')
print('=' * 60)

# --- 1. 인스턴스 타입 검증 ---
print('\n[1/4] 인스턴스 타입 검증')
training_instance = 'ml.p4d.24xlarge'
inference_instance = 'ml.g5.2xlarge'

train_ok = validate_instance_type(training_instance, 'training')
infer_ok = validate_instance_type(inference_instance, 'inference')
print(f'  학습 인스턴스 ({training_instance}): {"✅ 통과" if train_ok else "❌ 실패"}')
print(f'  추론 인스턴스 ({inference_instance}): {"✅ 통과" if infer_ok else "❌ 실패"}')

# 부적합한 인스턴스 예시
bad_ok = validate_instance_type('ml.g5.xlarge', 'training')
print(f'  ml.g5.xlarge (학습용): {"✅ 통과" if bad_ok else "❌ VRAM 부족 (정상)"}')

# --- 2. S3 URI 검증 ---
print('\n[2/4] S3 URI 형식 검증')
for name, uri in [('Dataset', S3_DATASET_URI), ('Model', S3_MODEL_URI), ('Output', S3_OUTPUT_URI)]:
    valid = validate_s3_uri(uri)
    print(f'  {name}: {"✅" if valid else "❌"} {uri}')

# --- 3. 인스턴스 추천 확인 ---
print('\n[3/4] 인스턴스 추천 목록')
print('  📋 학습용:')
for rec in get_instance_recommendations('training'):
    print(f'    - {rec["type"]} ({rec["gpu"]}, {rec["vram"]}) [{rec["tier"]}]')
print('  📋 추론용:')
for rec in get_instance_recommendations('inference'):
    print(f'    - {rec["type"]} ({rec["gpu"]}, {rec["vram"]}) [{rec["tier"]}]')

print('\n✅ 로컬 검증 완료')


### 5.1 추론 핸들러 로컬 테스트

SageMaker 추론 컨테이너 안에서 실행될 핸들러 함수들을 로컬에서 테스트합니다.
이렇게 하면 엔드포인트 배포 후 발생할 수 있는 입력 파싱/출력 직렬화 오류를 미리 잡을 수 있습니다.

테스트하는 함수:
- `input_fn` — JSON 요청 파싱 및 유효성 검증
- `predict_fn` — 모델 추론 (여기서는 stub)
- `output_fn` — 응답 JSON 직렬화


In [ ]:
from src.inference_handler import input_fn, predict_fn, output_fn, model_fn

print('🧪 추론 핸들러 로컬 테스트')
print('-' * 40)

# 테스트용 이미지를 base64로 인코딩
test_img = Image.new('RGB', (224, 224), (128, 64, 32))
img_buffer = io.BytesIO()
test_img.save(img_buffer, format='PNG')
test_image_b64 = base64.b64encode(img_buffer.getvalue()).decode('utf-8')

# 테스트 요청 생성
test_payload = {
    'image': test_image_b64,
    'proprioception': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
    'instruction': 'pick up the red block'
}

# 1. input_fn 테스트
parsed = input_fn(json.dumps(test_payload).encode(), 'application/json')
print(f'✅ input_fn: 파싱 성공')
print(f'   image: {len(parsed["image"])} chars (base64)')
print(f'   proprioception: {parsed["proprioception"]}')
print(f'   instruction: {parsed["instruction"]}')

# 2. predict_fn 테스트 (stub 모델 사용)
dummy_model = {'model_dir': '/tmp', 'loaded': True}
prediction = predict_fn(parsed, dummy_model)
print(f'\n✅ predict_fn: 추론 성공')
print(f'   actions: {prediction["actions"]}')
print(f'   timestamp: {prediction["timestamp"]}')

# 3. output_fn 테스트
output_bytes = output_fn(prediction, 'application/json')
output_parsed = json.loads(output_bytes)
print(f'\n✅ output_fn: 직렬화 성공')
print(f'   응답 크기: {len(output_bytes)} bytes')
print(f'   응답 내용: {output_parsed}')

# 4. 잘못된 입력 테스트 (에러 핸들링 확인)
print('\n🧪 에러 핸들링 테스트:')
try:
    input_fn(b'{"image": ""}', 'application/json')
except ValueError as e:
    print(f'  ✅ 빈 이미지 거부: {e}')

try:
    input_fn(b'{"image": "abc"}', 'text/plain')
except ValueError as e:
    print(f'  ✅ 잘못된 content-type 거부: {e}')

print('\n✅ 추론 핸들러 로컬 테스트 완료')


### 5.2 ECR 이미지 존재 확인

빌드한 컨테이너 이미지가 ECR에 정상적으로 푸시되었는지 확인합니다.
이미지가 없으면 SageMaker Training Job이 시작 시 실패합니다.


In [ ]:
# ECR에서 이미지 확인
ecr_client = boto3.client('ecr', region_name=REGION)

for repo_name, image_uri in [
    (ECR_TRAINING_REPO, TRAINING_IMAGE_URI),
    (ECR_INFERENCE_REPO, INFERENCE_IMAGE_URI)
]:
    try:
        response = ecr_client.describe_images(
            repositoryName=repo_name,
            imageIds=[{'imageTag': 'latest'}]
        )
        image_detail = response['imageDetails'][0]
        size_mb = image_detail.get('imageSizeInBytes', 0) / (1024 * 1024)
        pushed_at = image_detail.get('imagePushedAt', 'N/A')
        print(f'✅ {repo_name}: {size_mb:.0f} MB, 푸시 시각: {pushed_at}')
    except Exception as e:
        print(f'❌ {repo_name}: 이미지를 찾을 수 없음 - {e}')
        print(f'   → Step 4를 다시 실행하세요')


---
## Step 6: SageMaker Training Job 실행

이제 모든 준비가 완료되었습니다. SageMaker Training Job을 시작하여 GR00T-N1.6 모델을 파인튜닝합니다.

### Training Job이 하는 일

1. SageMaker가 지정된 인스턴스(예: `ml.p4d.24xlarge`)를 프로비저닝
2. ECR에서 학습 컨테이너를 풀(pull)
3. S3에서 모델 아티팩트와 데이터셋을 컨테이너 내부로 복사
   - 모델 → `/opt/ml/input/data/model/`
   - 데이터셋 → `/opt/ml/input/data/dataset/`
4. 컨테이너의 엔트리포인트(`train_entry.py`) 실행
5. 학습 완료 후 `/opt/ml/model/`의 내용을 S3에 자동 업로드

### 하이퍼파라미터

| 파라미터 | 값 | 설명 |
|---------|-----|------|
| `embodiment_tag` | `sample_robot` | 로봇 종류 식별자 (데이터셋의 robot_type과 매칭) |
| `max_steps` | `100` | 최대 학습 스텝 (테스트용으로 작게 설정) |
| `global_batch_size` | `4` | 배치 크기 (테스트용으로 작게 설정) |
| `save_steps` | `50` | 체크포인트 저장 간격 |

> 💡 **실제 학습 시**: `max_steps=10000`, `global_batch_size=32`로 설정하세요.
> 테스트 목적으로는 작은 값을 사용하여 빠르게 파이프라인을 검증합니다.

> ⚠️ **비용**: `ml.p4d.24xlarge`는 시간당 ~$32.77입니다.
> 테스트 시에는 `max_steps`를 작게 설정하여 비용을 최소화하세요.


In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

# 학습 하이퍼파라미터 (테스트용으로 작은 값 사용)
hyperparameters = {
    'embodiment_tag': 'sample_robot',
    'max_steps': '100',           # 테스트: 100 스텝 (실제: 10000)
    'global_batch_size': '4',     # 테스트: 4 (실제: 32)
    'save_steps': '50',           # 테스트: 50 (실제: 2000)
    'num_gpus': '1',
}

# SageMaker Estimator 생성
estimator = Estimator(
    image_uri=TRAINING_IMAGE_URI,
    role=ROLE_ARN,
    instance_type='ml.p4d.24xlarge',  # 8x A100 40GB (320GB VRAM)
    instance_count=1,
    output_path=S3_OUTPUT_URI,
    hyperparameters=hyperparameters,
    max_run=3600,                     # 최대 1시간 (안전장치)
    sagemaker_session=sm_session,
)

print('📋 Training Job 설정:')
print(f'  Image URI    : {TRAINING_IMAGE_URI}')
print(f'  Instance     : ml.p4d.24xlarge (8x A100, 320GB VRAM)')
print(f'  Output S3    : {S3_OUTPUT_URI}')
print(f'  Max Steps    : {hyperparameters["max_steps"]}')
print(f'  Batch Size   : {hyperparameters["global_batch_size"]}')


In [ ]:
# S3 입력 채널 설정
# SageMaker는 이 채널들을 컨테이너 내부의 지정된 경로에 마운트합니다:
#   'model'   → /opt/ml/input/data/model/   (SM_CHANNEL_MODEL)
#   'dataset' → /opt/ml/input/data/dataset/  (SM_CHANNEL_DATASET)
inputs = {
    'model': TrainingInput(s3_data=S3_MODEL_URI),
    'dataset': TrainingInput(s3_data=S3_DATASET_URI),
}

# Training Job 시작
print('🚀 Training Job 시작...')
print('   (CloudWatch 로그에서 실시간 진행 상황을 확인할 수 있습니다)')
print()

estimator.fit(inputs=inputs, wait=True, logs='All')

# 학습 완료 후 정보
TRAINING_JOB_NAME = estimator.latest_training_job.name
MODEL_ARTIFACT_S3 = estimator.model_data

print(f'\n✅ Training Job 완료!')
print(f'  Job Name       : {TRAINING_JOB_NAME}')
print(f'  Model Artifact : {MODEL_ARTIFACT_S3}')


### 6.1 학습 결과 확인

학습이 완료되면 모델 아티팩트가 `model.tar.gz`로 압축되어 S3에 저장됩니다.
이 파일이 이후 추론 엔드포인트 배포에 사용됩니다.


In [ ]:
# 학습 결과 S3 경로 확인
print(f'📦 모델 아티팩트 S3 경로: {MODEL_ARTIFACT_S3}')
print(f'\n📂 학습 출력 디렉토리:')
!aws s3 ls {S3_OUTPUT_URI}/ --recursive | head -20

# CloudWatch 로그 링크
log_url = (
    f'https://{REGION}.console.aws.amazon.com/cloudwatch/home'
    f'?region={REGION}#logsV2:log-groups/log-group/'
    f'%2Faws%2Fsagemaker%2FTrainingJobs/log-events/{TRAINING_JOB_NAME}'
)
print(f'\n📊 CloudWatch 로그: {log_url}')


---
## Step 7: SageMaker 추론 엔드포인트 배포

파인튜닝된 모델을 SageMaker 실시간 추론 엔드포인트로 배포합니다.

### 배포 과정

1. **SageMaker Model 생성** — ECR 이미지 + S3 모델 아티팩트를 연결
2. **EndpointConfig 생성** — 인스턴스 타입, 인스턴스 수 등 설정
3. **Endpoint 생성** — 실제 추론 서비스 시작

배포가 완료되면 HTTPS 엔드포인트가 생성되어 실시간 추론 요청을 받을 수 있습니다.

**추론 인스턴스 선택 기준:**
- GR00T-N1.6-3B는 최소 24GB GPU VRAM이 필요
- `ml.g5.2xlarge` (A10G 24GB)가 비용 대비 성능이 가장 좋음
- 대규모 배치 처리가 필요하면 `ml.p4d.24xlarge` 사용

> ⏱️ 엔드포인트 생성에 5-10분 소요됩니다.


In [ ]:
sm_client = boto3.client('sagemaker', region_name=REGION)
timestamp_short = datetime.utcnow().strftime('%Y%m%d%H%M%S')

MODEL_NAME = f'{ENDPOINT_NAME}-model-{timestamp_short}'
ENDPOINT_CONFIG_NAME = f'{ENDPOINT_NAME}-config-{timestamp_short}'
INFERENCE_INSTANCE_TYPE = 'ml.g5.2xlarge'  # 1x A10G 24GB

# 1. SageMaker Model 생성
# ECR 추론 이미지와 S3의 모델 아티팩트를 연결합니다.
print(f'📦 SageMaker Model 생성: {MODEL_NAME}')
sm_client.create_model(
    ModelName=MODEL_NAME,
    PrimaryContainer={
        'Image': INFERENCE_IMAGE_URI,
        'ModelDataUrl': MODEL_ARTIFACT_S3,
    },
    ExecutionRoleArn=ROLE_ARN,
)
print(f'  ✅ Model 생성 완료')

# 2. EndpointConfig 생성
# 어떤 인스턴스에서 몇 개를 실행할지 정의합니다.
print(f'⚙️  EndpointConfig 생성: {ENDPOINT_CONFIG_NAME}')
sm_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[{
        'VariantName': 'primary',
        'ModelName': MODEL_NAME,
        'InstanceType': INFERENCE_INSTANCE_TYPE,
        'InitialInstanceCount': 1,
    }],
)
print(f'  ✅ EndpointConfig 생성 완료')

# 3. Endpoint 생성
print(f'🚀 Endpoint 생성 시작: {ENDPOINT_NAME}')
print(f'   인스턴스: {INFERENCE_INSTANCE_TYPE} (1x A10G 24GB)')
sm_client.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
)
print(f'   생성 중... (5-10분 소요)')


In [ ]:
# 엔드포인트가 InService 상태가 될 때까지 대기
print(f'⏳ 엔드포인트 상태 모니터링: {ENDPOINT_NAME}')
print()

waiter = sm_client.get_waiter('endpoint_in_service')
try:
    waiter.wait(
        EndpointName=ENDPOINT_NAME,
        WaiterConfig={'Delay': 30, 'MaxAttempts': 40}  # 최대 20분 대기
    )
    print(f'\n✅ 엔드포인트 배포 완료!')
    print(f'   Endpoint Name: {ENDPOINT_NAME}')
    print(f'   Status: InService')
except Exception as e:
    # 현재 상태 확인
    desc = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = desc['EndpointStatus']
    reason = desc.get('FailureReason', 'N/A')
    print(f'\n❌ 엔드포인트 배포 실패')
    print(f'   Status: {status}')
    print(f'   Reason: {reason}')
    raise


---
## Step 8: 엔드포인트 호출 테스트

배포된 엔드포인트에 실제 추론 요청을 보내봅니다.

### 요청 형식

GR00T-N1.6 추론 엔드포인트는 다음 3가지 입력을 받습니다:

| 필드 | 타입 | 설명 |
|------|------|------|
| `image` | string (base64) | RGB 이미지를 base64로 인코딩한 문자열 |
| `proprioception` | list[float] | 로봇의 현재 관절 상태 벡터 |
| `instruction` | string | 자연어 작업 지시 (예: "pick up the red block") |

### 응답 형식

| 필드 | 타입 | 설명 |
|------|------|------|
| `actions` | list[list[float]] | 예측된 액션 벡터 시퀀스 |
| `timestamp` | string (ISO 8601) | 추론 시각 |


In [ ]:
# 테스트 이미지 생성 (224x224 RGB)
# 실제 사용 시에는 로봇 카메라에서 캡처한 이미지를 사용합니다.
test_image = Image.new('RGB', (224, 224), (100, 150, 200))

# 이미지를 base64로 인코딩
buffer = io.BytesIO()
test_image.save(buffer, format='PNG')
image_b64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

# 추론 요청 페이로드
payload = {
    'image': image_b64,
    'proprioception': [0.1, -0.2, 0.3, 0.0, 0.15, -0.05, 0.8],  # 7-DoF 관절 상태
    'instruction': 'pick up the red block and place it on the table'
}

print(f'📤 추론 요청 전송')
print(f'   Endpoint: {ENDPOINT_NAME}')
print(f'   Image: {len(image_b64)} chars (base64)')
print(f'   Proprioception: {payload["proprioception"]}')
print(f'   Instruction: "{payload["instruction"]}"')


In [ ]:
# SageMaker Runtime으로 엔드포인트 호출
runtime_client = boto3.client('sagemaker-runtime', region_name=REGION)

response = runtime_client.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Accept='application/json',
    Body=json.dumps(payload),
)

# 응답 파싱
result = json.loads(response['Body'].read().decode('utf-8'))

print('📥 추론 응답:')
print(f'   Timestamp: {result.get("timestamp", "N/A")}')
actions = result.get('actions', [])
print(f'   Actions ({len(actions)} steps):')
for i, action in enumerate(actions):
    print(f'     Step {i}: {action}')
    print(f'             → x={action[0]:.3f}, y={action[1]:.3f}, z={action[2]:.3f}, '
          f'rx={action[3]:.3f}, ry={action[4]:.3f}, rz={action[5]:.3f}, gripper={action[6]:.3f}')


### 8.1 여러 지시로 추론 테스트

다양한 자연어 지시로 모델의 응답을 확인합니다.
실제 파인튜닝된 모델에서는 지시에 따라 다른 액션 벡터가 출력됩니다.


In [ ]:
# 다양한 지시로 테스트
test_instructions = [
    'pick up the red block',
    'move to the left',
    'open the gripper',
    'place the object on the shelf',
]

print('🧪 다양한 지시로 추론 테스트')
print('=' * 60)

for instruction in test_instructions:
    payload['instruction'] = instruction
    
    response = runtime_client.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Accept='application/json',
        Body=json.dumps(payload),
    )
    result = json.loads(response['Body'].read().decode('utf-8'))
    actions = result.get('actions', [[]])[0]
    
    action_str = ', '.join(f'{a:.3f}' for a in actions)
    print(f'\n  📝 "{instruction}"')
    print(f'     → actions: [{action_str}]')

print('\n✅ 추론 테스트 완료')


---
## Step 9: 리소스 정리 ⚠️

**반드시 실행하세요!** SageMaker 엔드포인트는 실행 중인 동안 계속 비용이 발생합니다.

이 단계에서 정리하는 리소스:
1. **SageMaker Endpoint** — 추론 인스턴스 비용 중단
2. **EndpointConfig** — 설정 리소스 삭제
3. **SageMaker Model** — 모델 리소스 삭제

S3의 데이터와 ECR의 컨테이너 이미지는 삭제하지 않습니다.
이들은 이후 재사용할 수 있으며, 스토리지 비용은 상대적으로 매우 낮습니다.

> 💡 S3 데이터와 ECR 이미지도 삭제하려면 아래 '선택적 정리' 셀을 실행하세요.


In [ ]:
# 엔드포인트 삭제
print(f'🗑️  리소스 정리 시작')
print(f'   Endpoint: {ENDPOINT_NAME}')
print()

try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f'  ✅ Endpoint 삭제: {ENDPOINT_NAME}')
except Exception as e:
    print(f'  ⚠️  Endpoint 삭제 실패: {e}')

try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME)
    print(f'  ✅ EndpointConfig 삭제: {ENDPOINT_CONFIG_NAME}')
except Exception as e:
    print(f'  ⚠️  EndpointConfig 삭제 실패: {e}')

try:
    sm_client.delete_model(ModelName=MODEL_NAME)
    print(f'  ✅ Model 삭제: {MODEL_NAME}')
except Exception as e:
    print(f'  ⚠️  Model 삭제 실패: {e}')

print(f'\n✅ 리소스 정리 완료!')
print(f'   더 이상 추론 인스턴스 비용이 발생하지 않습니다.')


### 9.1 선택적 정리: S3 데이터 및 ECR 이미지 삭제

아래 셀은 **선택 사항**입니다. 테스트 데이터와 컨테이너 이미지까지 완전히 정리하고 싶을 때만 실행하세요.


In [ ]:
# ⚠️ 선택 사항: S3 데이터 삭제
# 주석을 해제하고 실행하면 S3의 테스트 데이터가 삭제됩니다.

# print('🗑️  S3 데이터 삭제')
# !aws s3 rm {S3_DATASET_URI} --recursive
# !aws s3 rm {S3_MODEL_URI} --recursive
# !aws s3 rm {S3_OUTPUT_URI} --recursive
# print('✅ S3 데이터 삭제 완료')

# ⚠️ 선택 사항: ECR 리포지토리 삭제
# !aws ecr delete-repository --repository-name {ECR_TRAINING_REPO} --region {REGION} --force
# !aws ecr delete-repository --repository-name {ECR_INFERENCE_REPO} --region {REGION} --force
# print('✅ ECR 리포지토리 삭제 완료')

print('💡 위 주석을 해제하고 실행하면 S3/ECR 리소스도 삭제됩니다.')


---
## 요약

이 노트북에서 수행한 전체 파이프라인:

```
[환경 설정] → [샘플 데이터 생성] → [S3 업로드] → [ECR 컨테이너 빌드]
     → [로컬 검증] → [SageMaker 학습] → [엔드포인트 배포]
     → [추론 테스트] → [리소스 정리]
```

### 실제 사용 시 변경할 부분

| 항목 | 이 노트북 (테스트) | 실제 사용 |
|------|-------------------|----------|
| 데이터셋 | 합성 샘플 (3 에피소드) | 실제 로봇 데이터 (수백~수천 에피소드) |
| 모델 | 더미 config.json | HuggingFace GR00T-N1.6-3B (~6GB) |
| max_steps | 100 | 10,000+ |
| global_batch_size | 4 | 32 |
| 학습 시간 | ~5분 | ~1-4시간 |

### 주요 변수 참조

이 노트북에서 생성된 주요 변수들:

```python
TRAINING_IMAGE_URI   # 학습 ECR 이미지 URI
INFERENCE_IMAGE_URI  # 추론 ECR 이미지 URI
S3_DATASET_URI       # 데이터셋 S3 경로
S3_MODEL_URI         # 베이스 모델 S3 경로
S3_OUTPUT_URI        # 학습 출력 S3 경로
MODEL_ARTIFACT_S3    # 학습 완료 후 모델 아티팩트 경로
TRAINING_JOB_NAME    # SageMaker Training Job 이름
ENDPOINT_NAME        # SageMaker Endpoint 이름
```

### 참고 자료

- [GR00T-N1.6 공식 문서](https://github.com/NVIDIA/Isaac-GR00T)
- [LeRobot 프로젝트](https://github.com/huggingface/lerobot)
- [SageMaker Training 문서](https://docs.aws.amazon.com/sagemaker/latest/dg/train-model.html)
- [SageMaker Endpoint 문서](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html)
- [프로젝트 가이드](../guide/GUIDE.md)
